# Lab 3 — Parallelization: sectioning + voting (§19.1.3)

Call the LLM **several times at once**, then let our code combine the results. The textbook splits this into 2 kinds:
1. **Sectioning (fan-out)** — split the work into non-overlapping parts and do each part separately
   Textbook example: code review 🔒 security ∥ ⚡ performance ∥ 🎨 style → merge · dedupe · rank by severity
   → total time ≈ the slowest call (**max**), not the total (**sum**)
2. **Voting** — the same prompt N times, then pick the answer by majority (or a reward model / LLM-as-judge)

Theme: the Cogito Learn teaching assistant helps review the code of a grading system

> Reference: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §19.1.3 (p.370) — CC BY-SA 4.0

> ✏️ **Starter notebook — you write the key parts**
>
> Most of the code is ready. Write only the **6 places marked `✏️ TODO`** (replace `___` or `...` with your code). Watch the video first, then try it yourself without opening the solution.
>
> Run every cell from top to bottom. A `NameError` or `TypeError` in a cell you have not finished yet is normal.
>
> Stuck? The solution is **`19-1-lab3-parallel.en.ipynb`** (the same code as in the video).

## 0. Setup

You need Python 3.10+ (or Colab) and an **OpenRouter API key** — setup instructions are in Lab 1

In [ ]:
%pip install -q "openai>=1.50" python-dotenv

In [ ]:
# ---- from Lab 1: set up client + ask_llm ----
import os, json
from openai import OpenAI

def load_key():
    try:
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        from dotenv import load_dotenv
        load_dotenv()
        return os.environ["OPENROUTER_API_KEY"]

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())
MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"provider": {"require_parameters": True}, "reasoning": {"enabled": False}}

def ask_llm(prompt, temperature=0.7, schema=None):
    kw = {"response_format": {"type": "json_schema", "json_schema": {"name": "out", "strict": True, "schema": schema}}} if schema else {}
    resp = client.chat.completions.create(model=MODEL, extra_body=EXTRA, temperature=temperature,
                                          messages=[{"role": "user", "content": prompt}], **kw)
    text = resp.choices[0].message.content
    return json.loads(text) if schema else text

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor
from collections import Counter

EXTRA["provider"]["sort"] = "throughput"

## 1. The code to review

The grading system's function that finds failing students (mock data for teaching — problems of several kinds planted on purpose)

In [ ]:
CODE = '''import sqlite3

DB_PASSWORD = "cogito123"

def find_failing(db, student_names):
    result = ""
    for n in student_names:
        q = "SELECT name, score FROM grades WHERE name = '" + n + "'"
        row = db.execute(q).fetchone()
        if row and row[1] < 50:
            if row[0] not in result.split(","):
                result = result + row[0] + ","
    return result
'''

NUMBERED = "\n".join(f"{i:2} | {line}" for i, line in enumerate(CODE.splitlines(), 1))
print(NUMBERED)

## 2. Sectioning: 3 reviewers

Each reviewer looks at **one aspect only**, with no overlap → answers in JSON `{line, severity, issue}`

In [ ]:
# ✏️ TODO 1: Define 3 reviewers (security / performance / style), each told to check only its own concern
REVIEWERS = {
    "security": "___",
    "performance": "___",
    "style": "___",
}

In [ ]:
FINDING = {"type": "object", "additionalProperties": False, "required": ["line", "severity", "issue"],
           "properties": {"line": {"type": "integer"},
                          "severity": {"type": "string", "enum": ["high", "medium", "low"]},
                          "issue": {"type": "string"}}}
SCHEMA = {"type": "object", "additionalProperties": False, "required": ["findings"],
          "properties": {"findings": {"type": "array", "minItems": 1, "items": FINDING}}}

In [ ]:
def review(aspect):
    start = time.perf_counter()
    # ✏️ TODO 2: Call the LLM with the prompt for aspect + the line-numbered code, using temperature 0 and SCHEMA
    out = ask_llm(___, temperature=___, schema=___)
    return aspect, out["findings"], time.perf_counter() - start

In [ ]:
def report(label, results, total):
    for aspect, findings, sec in results:
        print(f"   {aspect:12} {sec:5.1f}s  {len(findings)} findings")
    times = [sec for _, _, sec in results]
    print(f"⏱ {label}: {total:.1f}s  | sum of calls = {sum(times):.1f}s | slowest call = {max(times):.1f}s")

## 3. One call at a time vs. all at once

One at a time → total time ≈ **sum** · all at once → total time ≈ **max**

In [ ]:
start = time.perf_counter()
seq_results = [review(a) for a in REVIEWERS]
report("sequential", seq_results, time.perf_counter() - start)

In [ ]:
# ✏️ TODO 3: Send review for all 3 aspects at once with ThreadPoolExecutor and collect the results as a list
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=___) as pool:
    par_results = list(pool.___(review, REVIEWERS))
report("parallel", par_results, time.perf_counter() - start)

## 4. Merge · dedupe · rank by severity

Findings on **the same line** = one item (keep the highest severity + note which reviewers found it)

In [ ]:
SEVERITY = {"high": 0, "medium": 1, "low": 2}

def merge(results):
    by_line = {}
    for aspect, findings, _ in results:
        for f in findings:
            item = by_line.setdefault(f["line"], {"line": f["line"], "severity": "low", "by": [], "issues": []})
            if aspect not in item["by"]:
                item["by"].append(aspect)
            item["issues"].append(f["issue"])
            # ✏️ TODO 4: Keep the line's highest severity, then return the items sorted from high → low
            if ___:
                item["severity"] = f["severity"]
    return sorted(by_line.values(), key=lambda x: (___, x["line"]))

In [ ]:
merged = merge(par_results)
print(f"{sum(len(f) for _, f, _ in par_results)} findings → {len(merged)} items\n")
for m in merged:
    print(f"[{m['severity']:6}] line {m['line']:2}  ({' + '.join(m['by'])})")
    for issue in m["issues"]:
        print("      -", issue)

## 5. Voting: ask one question 5 times

same prompt · high temperature → each answer can differ → `Counter` counts → **majority**

In [ ]:
QUESTION = ("You review code for Cogito Learn, a university teaching-assistant app. "
            "Rate the severity of this code-review finding as high, medium or low.\n\n"
            "Finding: find_failing() returns a comma-separated string with a trailing comma "
            "(e.g. 'Ann,Bo,') instead of a list.")
VOTE = {"type": "object", "additionalProperties": False, "required": ["severity"],
        "properties": {"severity": {"type": "string", "enum": ["high", "medium", "low"]}}}

In [ ]:
# ✏️ TODO 5: Ask QUESTION once with a high temperature (1.0) and return the severity
def ask_once(_):
    return ask_llm(QUESTION, temperature=___, schema=VOTE)[___]

with ThreadPoolExecutor(max_workers=5) as pool:
    votes = list(pool.map(ask_once, range(5)))
print("votes:", votes)
# ✏️ TODO 6: Count the votes with Counter and take the answer with the most votes, plus its vote count
winner, count = ___
print(f"🗳️ majority: {winner}  ({count}/{len(votes)} agree)")

## Summary

| Kind | In the code | What you get |
|---|---|---|
| Sectioning (fan-out) | `REVIEWERS` + `review()` + `pool.map` | 3 aspects at once · time ≈ max, not sum |
| Merge | `merge()` | dedupe by line · sort by `SEVERITY` |
| Voting | `ask_once()` × 5 + `Counter` | majority + agreement count |

**Try next:** add a 4th reviewer such as `"tests"` to `REVIEWERS` and time it again — how much slower does sequential get, and how much slower does parallel get?
Also try changing the number of votes from 5 to 9 and see whether the agreement count gets steadier

**Next:** Lab 4 — Orchestrator-Workers